# NIH: more baselines and the relative-quality variant

Full resolution currently compares only FedAvg and AHA-Fed.


In [ ]:
import glob, os, shutil, sys
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist opacus


In [ ]:
import subprocess, sys
P=('import torch;'
   'a=torch.randn(4,3,8,8,device="cuda");'
   'w=torch.randn(2,3,3,3,device="cuda");'
   'torch.nn.functional.conv2d(a,w).sum().item();'
   'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',P],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',P],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


In [ ]:
import glob, os
csvs=glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert csvs, 'attach nih-chest-xrays/data'
CSV=csvs[0]; NIH_ROOT=os.path.dirname(CSV)
FLAT='/kaggle/working/nih_images'
os.makedirs(FLAT, exist_ok=True)
if len(os.listdir(FLAT)) < 100000:
    n=0
    for sh in sorted(glob.glob(f'{NIH_ROOT}/images_*/images')):
        for name in os.listdir(sh):
            dst=os.path.join(FLAT,name)
            if not os.path.exists(dst):
                os.symlink(os.path.join(sh,name), dst); n+=1
    print('linked', n)
print('images:', len(os.listdir(FLAT)))


In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from src.data.nih_chestxray import federate_nih_by_patient
from src.federated.client import FederatedClient
from src.federated.server import FederatedServer, build_aggregator
from src.models.backbones import build_model
from src.utils.seed import set_seed

NUM_CENTERS, NUM_CLASSES, NOISE_RATE = 4, 14, 0.5
IMAGES_PER_CENTER, ROUNDS = 2000, 15
device='cuda' if torch.cuda.is_available() else 'cpu'

centers=federate_nih_by_patient(csv_path=CSV, image_dir=FLAT,
                                num_centers=NUM_CENTERS)
noisy_centers=federate_nih_by_patient(
    csv_path=CSV, image_dir=FLAT, num_centers=NUM_CENTERS,
    label_noise=[0.0]*(NUM_CENTERS-1)+[NOISE_RATE])
eval_loader=DataLoader(Subset(centers[0], range(2000)), batch_size=64)
print('center sizes:', [len(c) for c in centers])


In [ ]:
import json
from pathlib import Path
OUT=Path('/kaggle/working/runs/paper/nih'); OUT.mkdir(parents=True, exist_ok=True)

def save(name, hist, cfg):
    payload={'config': dict(cfg, rounds_completed=len(hist)),
             'history': hist, 'final': hist[-1] if hist else None}
    (OUT/name).write_text(json.dumps(payload, indent=2, default=str))

def run(name, tag, seed, agg_kw, relative=False):
    f=OUT/name
    if f.exists() and json.loads(f.read_text())['config'].get(
            'rounds_completed',0) >= ROUNDS:
        print('[skip]', name); return
    set_seed(seed)
    shards=[Subset(c, range(min(IMAGES_PER_CENTER, len(c))))
            for c in noisy_centers]
    cl=[FederatedClient(i, ds, batch_size=32, device=device, multilabel=True)
        for i, ds in enumerate(shards)]
    agg=build_aggregator(**agg_kw)
    srv=FederatedServer(
        global_model=build_model('densenet121', num_classes=NUM_CLASSES,
                                 pretrained=True),
        clients=cl, aggregator=agg, eval_loader=eval_loader,
        num_classes=NUM_CLASSES, device=device, multilabel=True)
    cfg={'aggregator': tag, 'seed': seed, 'dataset': 'nih_chestxray14',
         'num_centers': NUM_CENTERS, 'images_per_center': IMAGES_PER_CENTER,
         'num_rounds': ROUNDS, 'noisy_client': NUM_CENTERS-1,
         'noisy_rate': NOISE_RATE, 'resolution': 224,
         'backbone': 'densenet121', 'relative_quality': relative}
    hist=[]
    srv.run(num_rounds=ROUNDS, local_epochs=1, local_lr=0.01, eval_every=1,
            round_callback=lambda m: (hist.append(m), save(name, hist, cfg)))
    print('[done]', name, '(%d rounds)' % len(hist))


In [ ]:
# Full resolution currently compares only FedAvg and AHA-Fed. Adding
# FedBN and FedDisco at three seeds is 6 runs x 1.7 h = 10.2 h, inside
# Kaggle's 12 h session limit. The relative-quality variant would add
# another 5 h and push past it, so it is left for a separate session.
import time
t0=time.time()
for seed in [42, 123, 456]:
    for tag, kw in [('fedbn', {'name': 'fedbn'}),
                    ('feddisco', {'name': 'feddisco', 'a': 4.0})]:
        if time.time()-t0 > 10.5*3600:
            print('stopping early to stay inside the session limit')
            break
        run('nih_noisy_%s_s%d.json' % (tag, seed), tag, seed, kw)
        print('  elapsed %.1f h' % ((time.time()-t0)/3600))


In [ ]:
!cd /kaggle/working && zip -qr nih_extra.zip runs/paper/nih
!ls -lh /kaggle/working/nih_extra.zip
